In [ ]:
# %%
# Cell - Imports
import os
import time
import random

import numpy as np
import pandas as pd
from numpy.linalg import eig
import scipy.io as sio

import tensorflow as tf
from tensorflow import Variable
from tensorflow import keras
from tensorflow.keras.models import Model
from keras.models import load_model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, Callback
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Input, Dense, Activation, Dropout, Conv1D, Conv2D, SeparableConv1D, SeparableConv2D, DepthwiseConv2D, AveragePooling2D, MaxPooling1D,
    BatchNormalization, Dropout, Add, Lambda,  Permute,
    Reshape, Concatenate, GlobalAveragePooling1D, GlobalMaxPooling1D,
    Multiply, Bidirectional, LSTM, Flatten
)
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.constraints import max_norm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

from scipy.signal import butter, filtfilt
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt


# --- Reproducibility ---
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("All imports successful.")
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# %%
# Cell - Pre-Processing & Data Loading
def load_BCI2a_data(data_path, subject, training, all_trials=True):
    n_channels = 22
    n_tests = 6 * 48
    window_Length = 7 * 250

    fs = 250
    t1 = int(2 * fs)
    t2 = int(6 * fs)

    class_return = np.zeros(n_tests)
    data_return = np.zeros((n_tests, n_channels, window_Length))

    NO_valid_trial = 0
    suffix = "T" if training else "E"
    file_path = os.path.join(data_path, f"A0{subject}{suffix}.mat")

    a = sio.loadmat(file_path)
    a_data = a['data']
    
    for ii in range(0, a_data.size):
        a_data1 = a_data[0, ii]
        a_data3 = a_data1[0, 0]
        a_X = a_data3[0]
        a_trial = a_data3[1]
        a_y = a_data3[2]
        a_artifacts = a_data3[5]

        for trial in range(0, a_trial.size):
            if a_artifacts[trial] != 0 and not all_trials:
                continue
            data_return[NO_valid_trial, :, :] = np.transpose(a_X[int(a_trial[trial]):(int(a_trial[trial]) + window_Length), :22])
            class_return[NO_valid_trial] = int(a_y[trial])
            NO_valid_trial += 1

    data_return = data_return[0:NO_valid_trial, :, t1:t2]
    class_return = class_return[0:NO_valid_trial]
    class_return = (class_return - 1).astype(int)

    return data_return, class_return

def get_data(path, subject):
    X_train, y_train = load_BCI2a_data(path, subject, True)
    X_test, y_test = load_BCI2a_data(path, subject, False)

    N_tr, N_ch, T = X_train.shape
    X_train = X_train.reshape(N_tr, N_ch, T)

    N_tr, N_ch, T = X_test.shape
    X_test = X_test.reshape(N_tr, N_ch, T)

    return X_train, y_train, X_test, y_test

In [ ]:
# %%
# Cell - Loading Subjects
data_path = rf"/kaggle/input/datasets/uddeshayanand/bci-iv-2a-dataset"

# subjects = [1, 2, 3, 4, 5, 6, 7, 8, 9]
subjects = [1]

# Lists to store data for all subjects
all_X_train = []
all_y_train = []
all_X_test = []
all_y_test = []

# Load data for each subject
for subject in subjects:
    print(subject)
    X_train, y_train, X_test, y_test = get_data(data_path, subject)

    # Append data to lists
    all_X_train.append(X_train)
    all_y_train.append(y_train)
    all_X_test.append(X_test)
    all_y_test.append(y_test)

# Combine data from all subjects
X_train = np.concatenate(all_X_train, axis=0)
y_train = np.concatenate(all_y_train, axis=0)
X_test = np.concatenate(all_X_test, axis=0)
y_test = np.concatenate(all_y_test, axis=0)

# Print the shapes
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# %%
# Cell - Augmentation
def sliding_window(data, sfreq=250, overlap=0.95, shift_sec=0.1):
    """
    Slice EEG trials into overlapping windows.
    Parameters
    ----------
    data : ndarray, shape (n_trials, n_chans, n_times)
    sfreq : float
        Sampling frequency in Hz.
    overlap : float
        Fractional overlap between windows.
    shift_sec : float
        Step size between window starts in seconds.
    Returns
    -------
    out : ndarray, shape (n_trials*n_windows, n_chans, window_length)
    """
    print(data.shape)
    n_trials, n_chans, n_times = data.shape
    shift = int(round(shift_sec * sfreq))                # e.g. 0.1s -> 25 samples
    window_length = int(round(shift / (1 - overlap)))    # e.g. 500 samples for 95% overlap
    n_windows = (n_times - window_length) // shift + 1    # number of windows per trial

    # Preallocate output array
    out = np.zeros((n_trials * n_windows, n_chans, window_length), dtype=data.dtype)
    idx = 0
    for tr in range(n_trials):
        for w in range(n_windows):
            start = w * shift
            stop = start + window_length
            out[idx] = data[tr, :, start:stop]
            idx += 1
    return out


# Apply sliding-window segmentation
X_train_seg = sliding_window(X_train, sfreq=250, overlap=0.95, shift_sec=0.1)
X_test_seg  = sliding_window(X_test,  sfreq=250, overlap=0.95, shift_sec=0.1)

# # Reshape back to 4D: (n_segments, 1, channels, window_length)
n_segs_train, n_ch, w_len = X_train_seg.shape
# X_train_seg = X_train_seg.reshape(n_segs_train, 1, n_ch, w_len)
n_segs_test, _, _ = X_test_seg.shape
# X_test_seg  = X_test_seg.reshape(n_segs_test, 1, n_ch, w_len)

# Generate segment-level labels
# Calculate number of windows per original trial
n_windows_train = n_segs_train // X_train.shape[0]
y_train_seg = np.repeat(y_train, n_windows_train)

n_windows_test = n_segs_test // X_test.shape[0]
y_test_seg  = np.repeat(y_test, n_windows_test)

# Print shapes for confirmation
print(f"Segmented X_train_seg shape: {X_train_seg.shape}, y_train_seg shape: {y_train_seg.shape}")
print(f"Segmented X_test_seg  shape: {X_test_seg.shape}, y_test_seg shape: {y_test_seg.shape}")

In [ ]:
# %%
# Cell - Splitting the data
X_trainS = X_train_seg.transpose(0, 2, 1)
X_testS = X_test_seg.transpose(0, 2, 1)

print("X_train shape:", X_trainS.shape)
print("X_test shape:", X_testS.shape)


# --- Prepare combined split ---
X_combined = np.concatenate((X_trainS, X_testS), axis=0)
y_combined = np.concatenate((y_train_seg, y_test_seg), axis=0)
print(X_combined.shape, y_combined.shape)

# --- Params ---
batch_size       = 64
epochs           = 2
validation_split = 0.1
test_split       = 0.2
num_classes      = 4


from sklearn.model_selection import train_test_split
X_trainval, X_test_new, y_trainval, y_test_new = train_test_split(
    X_combined, y_combined,
    test_size=test_split,
    stratify=y_combined,
    random_state=42
)
X_train_new, X_val_new, y_train_new, y_val_new = train_test_split(
    X_trainval, y_trainval,
    test_size=validation_split/(1-test_split),
    stratify=y_trainval,
    random_state=42
)


# --- One-hot encode labels ---
y_train_new_oh = to_categorical(y_train_new, num_classes)
y_val_new_oh   = to_categorical(y_val_new,   num_classes)
y_test_new_oh  = to_categorical(y_test_new,  num_classes)

In [ ]:
# %%
# Cell - Filtering
def apply_band_filters(X_data, fs=250, order=4):
    """
    Applies bandpass filtering to EEG data shaped (samples, timepoints, channels).
    Returns a tuple of filtered signals in different bands.
    """
    # 1. Remove DC offset across time axis (axis=1)
    X_data = X_data - np.mean(X_data, axis=1, keepdims=True)

    # 2. Band definitions
    bands = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 100)
    }

    # 3. Reshape to [samples * channels, timepoints] for filtering along time
    n_samples, n_time, n_channels = X_data.shape
    X_reshaped = X_data.transpose(0, 2, 1).reshape(-1, n_time)

    band_data = {}
    for band_name, (low, high) in bands.items():
        nyq = 0.5 * fs
        Wn = [low / nyq, high / nyq]

        if Wn[1] >= 1.0:
            raise ValueError(f"Invalid high cutoff {high}Hz for {fs}Hz sampling")

        b, a = butter(order, Wn, btype='bandpass')

        padlen = min(150, X_reshaped.shape[1] // 2)
        filtered = filtfilt(b, a, X_reshaped, axis=1, padlen=padlen)
        
        # Reshape back to original: (samples, time, channels)
        band_filtered = filtered.reshape(n_samples, n_time, n_channels).transpose(0, 2, 1)
        band_data[band_name] = band_filtered

    return (
        band_data['delta'],
        band_data['theta'],
        band_data['alpha'],
        band_data['beta'],
        band_data['gamma']
    )


# Apply band filtering
print("Processing bands...")

X_train_new = X_train_new.transpose(0, 2, 1)  # (3628, 22, 1000)
X_test_new = X_test_new.transpose(0, 2, 1)
X_val_new = X_val_new.transpose(0, 2, 1)

_, X_train_theta, X_train_alpha, X_train_beta, _ = apply_band_filters(X_train_new)

_, X_test_theta, X_test_alpha, X_test_beta, _ = apply_band_filters(X_test_new)

_, X_val_theta, X_val_alpha, X_val_beta, _ = apply_band_filters(X_val_new)


X_train_new = X_train_new.transpose(0, 2, 1)  # (3628, 22, 1000)
X_test_new = X_test_new.transpose(0, 2, 1)
X_val_new = X_val_new.transpose(0, 2, 1)

In [ ]:
# %%
# Cell - Model Functions

class Channel_attention(Model):
    def __init__(self,num_ch):
        super(Channel_attention,self).__init__(name='')
        self.conv1      = Conv2D(filters=8,kernel_size=(1,1),kernel_constraint = max_norm(2., axis=(0,1,2)))
        self.conv2      = Conv2D(filters=8,kernel_size=(1,1),kernel_constraint = max_norm(2., axis=(0,1,2)))
        self.reshape    = Reshape((num_ch,-1))
        self.gamma      = Variable(initial_value=0,dtype='float32',trainable=True)

    def call(self,x):
        proj_query      = self.reshape(self.conv1(x))
        proj_key        = self.reshape(self.conv2(x))

        energy          = tf.matmul(proj_query,proj_key,transpose_b=True)

        max_H           = tf.math.reduce_max(energy,axis=2,keepdims=True)
        min_H           = tf.math.reduce_min(energy,axis=2,keepdims=True)

        tmp_b           = (energy-min_H)
        tmp_c           = (max_H-min_H)+1e-8
        energy          = tmp_b/tmp_c
        attention       = tf.nn.softmax(energy)

        out = self.gamma * tf.expand_dims(tf.matmul(attention,tf.squeeze(x,axis=-1),transpose_a=True),axis=3) + x

        return out

class Time_attention(Model):
    def __init__(self,num_t):
        super(Time_attention,self).__init__(name='')
        self.conv1      = Conv2D(filters=8,kernel_size=(1,1),kernel_constraint = max_norm(2., axis=(0,1,2)))
        self.conv2      = Conv2D(filters=8,kernel_size=(1,1),kernel_constraint = max_norm(2., axis=(0,1,2)))
        self.reshape    = Reshape((num_t,-1))
        self.permute    = Permute((2,1,3))
        self.gamma      = Variable(initial_value=0,dtype='float32',trainable=True)

    def call(self,x):
        b,ch,t,d        = x.shape
        proj_query      = self.reshape(self.permute(self.conv1(x)))
        proj_key        = self.reshape(self.permute(self.conv2(x)))

        energy          = tf.matmul(proj_query,proj_key,transpose_b=True)

        max_H           = tf.math.reduce_max(energy,axis=2,keepdims=True)
        min_H           = tf.math.reduce_min(energy,axis=2,keepdims=True)

        tmp_b           = (energy-min_H)
        tmp_c           = (max_H-min_H)+1e-8
        energy          = tmp_b/tmp_c
        attention       = tf.nn.softmax(energy)

        out = self.gamma * tf.expand_dims(tf.matmul(tf.squeeze(x,axis=-1),attention,transpose_b=True),axis=3) + x

        return out

class MyAttention(Model):
    def __init__(self,num_ch,num_t):
        super(MyAttention,self).__init__()
        self.CA = Channel_attention(num_ch=num_ch)
        # self.TA = Time_attention(num_t=num_t)

    def call(self,x,training=None):
        out1 = self.CA(x)
        # out2 = self.TA(x)

        # out = tf.concat([out1, out2],axis=-1)

        return out1


def square(x):
    return K.square(x)


def log(x):
    return K.log(K.clip(x, min_value = 1e-7, max_value = 10000))


def TCN_block(input_layer,input_dimension,depth,kernel_size,filters,dropout,activation='relu'):
    block = Conv1D(filters,kernel_size=kernel_size,dilation_rate=1,activation='linear',
                   padding = 'causal',kernel_initializer='he_uniform')(input_layer)
    block = BatchNormalization()(block)
    block = Activation(activation)(block)
    block = Dropout(dropout)(block)
    block = Conv1D(filters,kernel_size=kernel_size,dilation_rate=1,activation='linear',
                   padding = 'causal',kernel_initializer='he_uniform')(block)
    block = BatchNormalization()(block)
    block = Activation(activation)(block)
    block = Dropout(dropout)(block)
    if(input_dimension != filters):
        conv = Conv1D(filters,kernel_size=1,padding='same')(input_layer)
        added = Add()([block,conv])
    else:
        added = Add()([block,input_layer])
    out = Activation(activation)(added)

    for i in range(depth-1):
        block = Conv1D(filters,kernel_size=kernel_size,dilation_rate=2**(i+1),activation='linear',
                   padding = 'causal',kernel_initializer='he_uniform')(out)
        block = BatchNormalization()(block)
        block = Activation(activation)(block)
        block = Dropout(dropout)(block)
        block = Conv1D(filters,kernel_size=kernel_size,dilation_rate=2**(i+1),activation='linear',
                   padding = 'causal',kernel_initializer='he_uniform')(block)
        block = BatchNormalization()(block)
        block = Activation(activation)(block)
        block = Dropout(dropout)(block)
        added = Add()([block, out])
        out = Activation(activation)(added)

    return out


def cbam_block_1d(input_feature, ratio=8, residual=False):
    cbam_feature = channel_attention_1d(input_feature, ratio)
    cbam_feature = spatial_attention_1d(cbam_feature)

    if residual:
        cbam_feature = Add()([input_feature, cbam_feature])
    return cbam_feature

def channel_attention_1d(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    shared_layer_one = Dense(channel // ratio,
                             activation='relu',
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')
    shared_layer_two = Dense(channel,
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')

    avg_pool = GlobalAveragePooling1D()(input_feature)
    avg_pool = Reshape((1, channel))(avg_pool)
    avg_pool = shared_layer_one(avg_pool)
    avg_pool = shared_layer_two(avg_pool)

    max_pool = GlobalMaxPooling1D()(input_feature)
    max_pool = Reshape((1, channel))(max_pool)
    max_pool = shared_layer_one(max_pool)
    max_pool = shared_layer_two(max_pool)

    cbam_feature = Add()([avg_pool, max_pool])
    cbam_feature = Activation('sigmoid')(cbam_feature)

    return Multiply()([input_feature, cbam_feature])

def spatial_attention_1d(input_feature):
    avg_pool = Lambda(lambda x: K.mean(x, axis=2, keepdims=True))(input_feature)  # shape: (batch, time, 1)
    max_pool = Lambda(lambda x: K.max(x, axis=2, keepdims=True))(input_feature)   # shape: (batch, time, 1)

    concat = Concatenate(axis=2)([avg_pool, max_pool])  # shape: (batch, time, 2)

    cbam_feature = Conv1D(filters=1,
                          kernel_size=7,
                          padding='same',
                          activation='sigmoid',
                          kernel_initializer='he_normal',
                          use_bias=False)(concat)

    return Multiply()([input_feature, cbam_feature])
    
class GatedAttentionFusion(tf.keras.layers.Layer):
    def __init__(self, fused_dim=64):
        super().__init__()
        self.fused_dim = fused_dim

        # Projection layers for the two inputs
        self.proj1 = tf.keras.layers.Dense(fused_dim)
        self.proj2 = tf.keras.layers.Dense(fused_dim)

        # Attention layers
        self.attn_dense = tf.keras.layers.Dense(fused_dim, activation='tanh')
        self.gate_score = tf.keras.layers.Dense(1)

    def call(self, x1, x2):
        # Project inputs to same dimension
        p1 = self.proj1(x1)
        p2 = self.proj2(x2)

        # Stack along source dimension -> (B, T, 2, D)
        stacked = tf.stack([p1, p2], axis=2)

        # Attention scores
        attn_hidden = self.attn_dense(stacked)
        attn_scores = self.gate_score(attn_hidden)

        # Attention weights across sources
        attn_weights = tf.nn.softmax(attn_scores, axis=2)

        # Weighted fusion
        fused = tf.reduce_sum(attn_weights * stacked, axis=2)

        return fused    

In [ ]:
# %%
# Cell - Build the Model
temporal_dimension = X_train_new.shape[1]
num_channels = X_train_new.shape[2]
num_classes = 4

# Ablation 1: No Theta — Sub-band fusion: Alpha + Beta only
input_main  = Input(shape=(temporal_dimension, num_channels), name="input_main")
input_alpha = Input(shape=(temporal_dimension, num_channels), name="input_alpha")
input_beta  = Input(shape=(temporal_dimension, num_channels), name="input_beta")

# --- Main Branch ---
x_main = Reshape((num_channels, temporal_dimension, 1))(input_main)
x_main = MyAttention(num_ch=num_channels, num_t=temporal_dimension)(x_main)
x      = Reshape((temporal_dimension, num_channels))(x_main)
x1     = TCN_block(input_layer=input_main, input_dimension=16, depth=2,
                   kernel_size=4, filters=22, dropout=0.3, activation='elu')
x2     = tf.keras.layers.Concatenate()([x, x1])
x3     = cbam_block_1d(x2)

# --- Alpha Branch ---
x_alpha = Reshape((num_channels, temporal_dimension, 1))(input_alpha)
x_alpha = MyAttention(num_ch=num_channels, num_t=temporal_dimension)(x_alpha)
xa      = Reshape((temporal_dimension, num_channels))(x_alpha)

# --- Beta Branch ---
x_beta = Reshape((num_channels, temporal_dimension, 1))(input_beta)
x_beta = MyAttention(num_ch=num_channels, num_t=temporal_dimension)(x_beta)
xb     = Reshape((temporal_dimension, num_channels))(x_beta)

# --- Fusion (Alpha + Beta only) ---
x2_sub = tf.keras.layers.Concatenate()([xa, xb])
x31    = cbam_block_1d(x2_sub)

fused_dim = int(x3.shape[-1]) + int(x31.shape[-1])
fused     = GatedAttentionFusion(fused_dim=fused_dim)(x3, x31)

# --- Output Block (same for all variants) ---
out = Conv1D(32, 5, activation='relu', padding='same')(fused)
out = MaxPooling1D(2)(out)
out = BatchNormalization()(out)
out = Dropout(0.4)(out)
out = Conv1D(64, 3, activation='relu', padding='same')(out)
out = BatchNormalization()(out)
out = GlobalAveragePooling1D()(out)
out = Dropout(0.3)(out)
out = Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.002))(out)
out = Dense(units=4)(out)
out = Activation('softmax')(out)

model_no_theta = Model(inputs=[input_main, input_alpha, input_beta], outputs=out)

model_no_theta.build(((None, temporal_dimension, num_channels)))
model_no_theta.summary()

In [ ]:
# %%
# Cell -
model_no_theta.compile(loss='categorical_crossentropy',
                  optimizer=Adam(learning_rate=1e-4),
                  metrics=['accuracy',tf.keras.metrics.Precision(),
                           tf.keras.metrics.Recall()
                                                                       ])

# --- Custom threshold callback ---
class ThresholdStopping(Callback):
    def __init__(self, monitor='val_loss', threshold=0.0254):
        super().__init__()
        self.monitor = monitor
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is not None and current <= self.threshold:
            print(f"\nEpoch {epoch+1}: {self.monitor} reached {current:.4f} Ã¢â€°Â¤ {self.threshold}, stopping.")
            self.model.stop_training = True

# 2. Ensure results directory exists
results_dir = "STSPecResult_A7S1"
os.makedirs(results_dir, exist_ok=True)


checkpoint = [
    ThresholdStopping(monitor='val_loss', threshold=0.0254),
    ModelCheckpoint(
filepath=os.path.join(results_dir,'STSPecbest_model_A7S1.h5'),
                    monitor='val_loss',
                    save_best_only=True,
                    mode='min'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=20, min_lr=1e-5)
]


# --- Train ---
print("\nStarting training...")
start_time = time.time()
history = model_no_theta.fit([X_train_new, X_train_alpha, X_train_beta], 
                        y_train_new_oh,epochs=1320,
                        batch_size=64,
                        validation_data=([X_val_new, 
                                          X_val_alpha, X_val_beta], y_val_new_oh),
                        callbacks = checkpoint,
                        shuffle=True, verbose=2)

total_time = time.time() - start_time
print(f"\nTraining completed in {int(total_time//60)}m {total_time%60:.1f}s")

In [ ]:
# --- Function to compute and print metrics ---
def evaluate_split(X, y_true, split_name="Test"):
    y_prob = model_no_theta.predict(X)
    y_pred = np.argmax(y_prob, axis=1)
    acc   = accuracy_score(y_true, y_pred)
    prec  = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec   = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1    = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"\n*** {split_name} Metrics ***")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"Kappa    : {kappa:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))
    return confusion_matrix(y_true, y_pred)


# --- Evaluate on Train, Validation, Test ---
cm_train = evaluate_split([X_train_new, X_train_alpha, X_train_beta], y_train_new, split_name="Training")
cm_val   = evaluate_split([X_val_new, X_val_alpha, X_val_beta],   y_val_new,   split_name="Validation")
cm_test  = evaluate_split([X_test_new, X_test_alpha, X_test_beta],  y_test_new,  split_name="Test")


def plot_confusion_matrix_percent(cm, class_labels, title, save_path):
    # Convert to float for percentage calculation
    cm = cm.astype('float')

    # Safe row-wise normalization: avoid division by zero
    row_sums = cm.sum(axis=1, keepdims=True)
    # Replace zeros in row_sums with 1 to avoid division by zero
    row_sums[row_sums == 0] = 1

    cm_percent = (cm / row_sums) * 100

    # Create formatted string labels
    labels = np.array([["{:.1f}%".format(val) for val in row] for row in cm_percent])

    # Plotting
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(cm_percent, interpolation='nearest', cmap=plt.cm.Greens)
    ax.figure.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(len(class_labels)),
        yticks=np.arange(len(class_labels)),
        xticklabels=class_labels,
        yticklabels=class_labels,
        xlabel='Predicted label',
        ylabel='True label',
       # title=title
    )

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    # Annotate with percentage text
    thresh = cm_percent.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, labels[i, j],
                    ha="center", va="center",
                    color="white" if cm_percent[i, j] > thresh else "black")

    fig.tight_layout()
    fig.savefig(save_path)
    plt.close(fig)

classes_labels = ['Left hand', 'Right hand', 'Foot', 'Tongue']
# --- Plot & save confusion matrices ---
for cm, name in zip([cm_train, cm_val, cm_test], ["Train", "Val", "Test"]):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes_labels)
    fig, ax = plt.subplots(figsize=(5,5))
    disp.plot(cmap=plt.cm.Blues, ax=ax)
    # ax.set_title(f"{name} Confusion Matrix")
    fig.tight_layout()
    fig.savefig(os.path.join(results_dir,f"{name}_confusion_matrix_A6S1.png"))
  #  plt.show()

for cm, name in zip([cm_train, cm_val, cm_test], ["Train", "Val", "Test"]):
    save_path = os.path.join(results_dir, f"{name}_confusion_matrix_percent_A7S1.png")
    plot_confusion_matrix_percent(cm, classes_labels, f"{name} A6S1Confusion Matrix (Percentages)", save_path)


# --- Plot & save learning curves ---
fig, axs = plt.subplots(1, 2, figsize=(12,5))
axs[0].plot(history.history['accuracy'])
axs[0].plot(history.history['val_accuracy'])
axs[0].set_title('Accuracy')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Accuracy')
axs[0].legend(['Train','Val'], loc='upper left')

axs[1].plot(history.history['loss'])
axs[1].plot(history.history['val_loss'])
axs[1].set_title('Loss')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Loss')
axs[1].legend(['Train','Val'], loc='upper left')

fig.tight_layout()
fig.savefig(os.path.join(results_dir, "learning_curves_STSPec_A7S1.png"))
plt.show()


# Flatten the EEG data: assuming shape (1000, channels, time) Ã¢â€ â€™ (1000, features)
features = model_no_theta.predict([X_train_new,
                          X_train_alpha, X_train_beta])
X_flat = features.reshape(features.shape[0], -1)

# Run t-SNE
tsne = TSNE(n_components=2, random_state=42)
embeddings_train = tsne.fit_transform(X_flat)

# Unique class labels (e.g., [0, 1, 2, 3])
unique_classes = np.unique(y_train_new)
print(unique_classes)
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_classes)))  # Choose a color map

# Plot each class separately
plt.figure(figsize=(10, 8))
for i, cls in enumerate(unique_classes):
    idx = y_train_new == cls
    plt.scatter(embeddings_train[idx, 0], embeddings_train[idx, 1],
                color=colors[i], label=f'Class {cls}', s=40)

plt.title('t-SNE Plot of EEG Data Features')
plt.xlabel('t-SNE Feature 1')
plt.ylabel('t-SNE Feature 2')
plt.legend(title='Classes')
plt.grid(True)
plt.tight_layout()
plt.show()


# --- Plot & save learning curves ---
fig, axs = plt.subplots(1,1, figsize=(8,5))
# Unique class labels (e.g., [0, 1, 2, 3])
unique_classes = np.unique(y_train_new)
classes_labels = ['Left hand', 'Right hand', 'Foot', 'Tongue']
for i, cls in enumerate(unique_classes):
    idx = y_train_new == cls
    plt.scatter(embeddings_train[idx, 0], embeddings_train[idx, 1],
                color=colors[i], label=f'{classes_labels[i]}', s=40)
# axs[0].set_title('Accuracy')
axs.set_xlabel('t-SNE Feature 1')
axs.set_ylabel('t-SNE Feature 2')
axs.legend(title='Classes', loc='upper left')

fig.tight_layout()
fig.savefig(os.path.join(results_dir, "Train_STSPecTSNE_A7S1.png"))
plt.show()


####################### Validation tsne #####################################

features = model_no_theta.predict([X_val_new, X_val_alpha, X_val_beta])
X_flat = features.reshape(features.shape[0], -1)

# Run t-SNE
tsne = TSNE(n_components=2, random_state=42)
embeddings_val = tsne.fit_transform(X_flat)

# Unique class labels (e.g., [0, 1, 2, 3])
unique_classes = np.unique(y_val_new)
print(unique_classes)
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_classes)))  # Choose a color map

# Plot each class separately
plt.figure(figsize=(10, 8))
for i, cls in enumerate(unique_classes):
    idx = y_val_new == cls
    plt.scatter(embeddings_val[idx, 0], embeddings_val[idx, 1],
                color=colors[i], label=f'Class {cls}', s=40)

plt.title('t-SNE Plot of EEG Data Features')
plt.xlabel('t-SNE Feature 1')
plt.ylabel('t-SNE Feature 2')
plt.legend(title='Classes')
plt.grid(True)
plt.tight_layout()

# --- Plot & save learning curves ---
fig, axs = plt.subplots(1,1, figsize=(8,5))
# Unique class labels (e.g., [0, 1, 2, 3])
unique_classes = np.unique(y_val_new)
classes_labels = ['Left hand', 'Right hand', 'Foot', 'Tongue']
for i, cls in enumerate(unique_classes):
    idx = y_val_new == cls
    plt.scatter(embeddings_val[idx, 0], embeddings_val[idx, 1],
                color=colors[i], label=f'{classes_labels[i]}', s=40)
# axs[0].set_title('Accuracy')
axs.set_xlabel('t-SNE Feature 1')
axs.set_ylabel('t-SNE Feature 2')
axs.legend(title='Classes', loc='upper left')

fig.tight_layout()
fig.savefig(os.path.join(results_dir, "Val_STSPecTSNE_A7S1.png"))


####################### Testing tsne   #######################################

features = model_no_theta.predict([X_test_new, X_test_alpha, X_test_beta])
X_flat = features.reshape(features.shape[0], -1)

# Run t-SNE
tsne = TSNE(n_components=2, random_state=42)
embeddings_test = tsne.fit_transform(X_flat)

# Unique class labels (e.g., [0, 1, 2, 3])
unique_classes = np.unique(y_test_new)
print(unique_classes)
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_classes)))  # Choose a color map

# Plot each class separately
plt.figure(figsize=(10, 8))
for i, cls in enumerate(unique_classes):
    idx = y_test_new == cls
    plt.scatter(embeddings_test[idx, 0], embeddings_test[idx, 1],
                color=colors[i], label=f'Class {cls}', s=40)

plt.title('t-SNE Plot of EEG Data Features')
plt.xlabel('t-SNE Feature 1')
plt.ylabel('t-SNE Feature 2')
plt.legend(title='Classes')
plt.grid(True)
plt.tight_layout()

# --- Plot & save learning curves ---
fig, axs = plt.subplots(1,1, figsize=(8,5))
# Unique class labels (e.g., [0, 1, 2, 3])
unique_classes = np.unique(y_test_new)
classes_labels = ['Left hand', 'Right hand', 'Foot', 'Tongue']
for i, cls in enumerate(unique_classes):
    idx = y_test_new == cls
    plt.scatter(embeddings_test[idx, 0], embeddings_test[idx, 1],
                color=colors[i], label=f'{classes_labels[i]}', s=40)
# axs[0].set_title('Accuracy')
axs.set_xlabel('t-SNE Feature 1')
axs.set_ylabel('t-SNE Feature 2')
axs.legend(title='Classes', loc='upper left')

fig.tight_layout()
fig.savefig(os.path.join(results_dir, "Test_STSPecTSNE_A7S1.png"))


# Save full history
np.savez(
    os.path.join(results_dir, "training_history4Va_A7S1.npz"),
    **history.history
)

print(f"\nAll results saved")